# singular-matrix-mask-trick — worked example 3: Count and retrieve valid solutions from a mixed batch

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `singular-matrix-mask-trick`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

After applying the singular mask trick, you can use the `is_valid` boolean tensor as an index to extract only the meaningful solutions. `x[is_valid]` gives shape (num_valid, n), and `is_valid.sum()` tells you how many systems were actually solved. This pattern is common in ray-tracing pipelines where many rays miss all geometry.

## Worked solution

**Step 1 — Build a batch of 6 systems where 2 are singular.** We track which indices are supposed to be singular so we can verify the mask.

**Step 2 — Apply the mask trick.** Standard procedure: det → is_singular → clone → patch → solve.

**Step 3 — Count valid solves.** `is_valid.sum()` should be 4.

**Step 4 — Verify residuals.** For each valid system, `A[i] @ x[i] - b[i]` should be near zero, confirming the solve was correct (not just that we detected it as valid).

In [ ]:
import torch as t

t.manual_seed(99)

def masked_batch_solve(A, b, eps=1e-8):
    K, n, _ = A.shape
    dets = t.linalg.det(A)
    is_singular = dets.abs() < eps
    A_safe = A.clone()
    A_safe[is_singular] = t.eye(n, dtype=A.dtype)
    x = t.linalg.solve(A_safe, b)
    return x, ~is_singular

t.manual_seed(99)
# Build 6 systems: indices 2 and 4 are singular
A = t.eye(2).unsqueeze(0).repeat(6, 1, 1)  # start with 6 identities
A = A + t.randn(6, 2, 2) * 0.3            # perturb to make them non-trivial
# Force indices 2 and 4 to be singular
A[2, 1] = A[2, 0] * 2.1   # row1 = 2.1*row0 -> singular
A[4, 1] = A[4, 0] * -0.5  # row1 = -0.5*row0 -> singular
b = t.randn(6, 2)

x, valid = masked_batch_solve(A, b)
print(f'Valid count: {valid.sum().item()} of 6  (expected 4)')
print(f'Valid mask: {valid.tolist()}')
for i in range(6):
    if valid[i]:
        res = (A[i] @ x[i] - b[i]).abs().max().item()
        print(f'  slice {i}: residual = {res:.2e}')